In [3]:
import pandas as pd
import numpy as np
import joblib
import optuna
from optuna.samplers import TPESampler
from datetime import datetime
from darts import TimeSeries
from darts.models import RandomForestModel, RegressionModel, XGBModel, LightGBMModel
from darts.dataprocessing.transformers import Scaler, Diff
from darts.utils.missing_values import fill_missing_values
from sklearn.ensemble import ExtraTreesRegressor
import warnings
import gc

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Load data dari notebook utama (yang sudah di-save)
df_merged = joblib.load("saved_models/df_merged_20260408_1820.joblib")

LEVEL_VARS = ["M2", "USDIDR", "Coal", "Copper", "Nickel", "Silver", "Tin", "STI", "Gold"]
RATE_VARS = ["BI_Rate", "CPI", "NPL_Ratio"]

print(f"Data loaded: {df_merged.shape}")
print(f"Date range: {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")


Data loaded: (2443, 14)
Date range: 2015-01-02 to 2025-01-31


In [4]:
def to_series(df, target_col, covariates=None):
    """Convert DataFrame to Darts TimeSeries."""
    target_series = TimeSeries.from_dataframe(
        df, time_col="date", value_cols=target_col,
        fill_missing_dates=True, freq="B"
    )
    target_series = fill_missing_values(target_series)
    covariate_series = None
    if covariates:
        covariate_series = TimeSeries.from_dataframe(
            df, time_col="date", value_cols=covariates,
            fill_missing_dates=True, freq="B"
        )
        covariate_series = fill_missing_values(covariate_series)
    return target_series, covariate_series


def expanding_cv_score(model_builder, target_series, n_folds=5):
    """
    Run expanding window CV, return list of MAPE per fold.
    model_builder: callable that returns a fresh model instance.
    """
    n = len(target_series)
    test_size = int(n * 0.15)
    min_train = int(n * 0.4)
    available = n - min_train - test_size
    step = max(1, available // (n_folds - 1))
    
    mape_scores = []
    
    for fold in range(n_folds):
        train_end_idx = min_train + fold * step
        test_end_idx = min(train_end_idx + test_size, n)
        if test_end_idx > n:
            break
        
        train_ts = target_series[:train_end_idx]
        test_ts = target_series[train_end_idx:test_end_idx]
        fold_ts = target_series[:test_end_idx]
        
        # Log-diff
        fold_log = fold_ts.map(np.log)
        train_log = train_ts.map(np.log)
        differencer = Diff(lags=1)
        train_log_diff = differencer.fit_transform(train_log)
        fold_log_diff = differencer.transform(fold_log)
        
        # Scale
        scaler = Scaler()
        train_scaled = scaler.fit_transform(train_log_diff)
        fold_scaled = scaler.transform(fold_log_diff)
        
        # Train
        model = model_builder()
        model.fit(train_scaled)
        
        # Predict
        forecast_list = model.historical_forecasts(
            series=fold_scaled,
            start=test_ts.start_time(),
            forecast_horizon=1,
            stride=1,
            retrain=False,
            last_points_only=False,
            verbose=False
        )
        if isinstance(forecast_list, TimeSeries):
            forecast_list = [forecast_list]
        
        # Inverse transform
        all_pred_dates, all_pred_prices = [], []
        fold_log_full = fold_ts.map(np.log)
        for chunk in forecast_list:
            chunk_diff = scaler.inverse_transform(chunk)
            dates = chunk_diff.time_index
            vals = chunk_diff.values().flatten()
            idx = fold_ts.get_index_at_point(dates[0])
            anchor = fold_log_full[idx - 1].values()[0][0]
            prices = np.exp(anchor + np.cumsum(vals))
            all_pred_dates.extend(dates)
            all_pred_prices.extend(prices)
        
        pred_df = pd.DataFrame({"date": pd.to_datetime(all_pred_dates), "predicted": all_pred_prices})
        actual_df = fold_ts.to_dataframe().reset_index()
        actual_df.columns = ["date", "actual"]
        eval_df = pd.merge(actual_df, pred_df, on="date", how="inner")
        
        y_true = eval_df["actual"].values
        y_pred = eval_df["predicted"].values
        mape_f = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        mape_scores.append(mape_f)
        
        del model
        gc.collect()
    
    return mape_scores

# Prepare target series (univariate, no covariates — for tuning speed)
target_series, _ = to_series(df_merged, "IHSG")
print(f"Target series length: {len(target_series)}")
print("Functions ready.")


Target series length: 2631
Functions ready.


In [5]:
WINDOW = 120

def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]),
        "max_samples": trial.suggest_float("max_samples", 0.5, 0.9, step=0.1),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
    }
    def builder():
        return RandomForestModel(
            lags=WINDOW, output_chunk_length=1,
            random_state=42, n_jobs=-1, **params
        )
    scores = expanding_cv_score(builder, target_series, n_folds=3)
    return np.mean(scores)


def et_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
    }
    def builder():
        return RegressionModel(
            lags=WINDOW, output_chunk_length=1,
            model=ExtraTreesRegressor(random_state=42, n_jobs=-1, **params)
        )
    scores = expanding_cv_score(builder, target_series, n_folds=3)
    return np.mean(scores)


def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    }
    def builder():
        return XGBModel(
            lags=WINDOW, output_chunk_length=1,
            random_state=42, n_jobs=-1, **params
        )
    scores = expanding_cv_score(builder, target_series, n_folds=3)
    return np.mean(scores)


def lgbm_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
    }
    def builder():
        return LightGBMModel(
            lags=WINDOW, output_chunk_length=1,
            random_state=42, n_jobs=-1, verbose=-1, **params
        )
    scores = expanding_cv_score(builder, target_series, n_folds=3)
    return np.mean(scores)


OBJECTIVES = {
    "RandomForest": rf_objective,
    "ExtraTrees": et_objective,
    "XGBoost": xgb_objective,
    "LightGBM": lgbm_objective,
}

print("Objective functions defined.")


Objective functions defined.


In [6]:
N_TRIALS = 50  # bisa naikkan ke 100 kalau waktu cukup

TUNING_RESULTS = {}

for model_name, objective_fn in OBJECTIVES.items():
    print(f"\n{'='*60}")
    print(f"Tuning {model_name} ({N_TRIALS} trials)")
    print(f"{'='*60}")
    
    sampler = TPESampler(seed=42)
    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        study_name=f"IHSG_{model_name}",
    )
    
    start = datetime.now()
    study.optimize(objective_fn, n_trials=N_TRIALS, show_progress_bar=True)
    elapsed = datetime.now() - start
    
    TUNING_RESULTS[model_name] = {
        "best_params": study.best_params,
        "best_value": study.best_value,
    }
    
    print(f"\n{model_name} done in {elapsed}")
    print(f"  Best MAPE: {study.best_value:.4f}%")
    print(f"  Best params: {study.best_params}")

# Save
joblib.dump(TUNING_RESULTS, "saved_models/optuna_tuning_results.joblib")
print("\nSaved: saved_models/optuna_tuning_results.joblib")



Tuning RandomForest (50 trials)


Best trial: 41. Best value: 0.633631: 100%|██████████| 50/50 [13:04<00:00, 15.68s/it]



RandomForest done in 0:13:04.331955
  Best MAPE: 0.6336%
  Best params: {'n_estimators': 800, 'max_depth': 3, 'max_features': 'log2', 'max_samples': 0.7, 'min_samples_split': 9, 'min_samples_leaf': 7}

Tuning ExtraTrees (50 trials)


Best trial: 44. Best value: 0.632949: 100%|██████████| 50/50 [04:02<00:00,  4.86s/it]



ExtraTrees done in 0:04:02.970681
  Best MAPE: 0.6329%
  Best params: {'n_estimators': 500, 'max_depth': 5, 'max_features': 'log2', 'min_samples_split': 4, 'min_samples_leaf': 9}

Tuning XGBoost (50 trials)


Best trial: 24. Best value: 0.632919: 100%|██████████| 50/50 [22:35<00:00, 27.12s/it] 



XGBoost done in 0:22:35.959148
  Best MAPE: 0.6329%
  Best params: {'n_estimators': 300, 'max_depth': 11, 'learning_rate': 0.014782371552985231, 'subsample': 0.8551536361803161, 'colsample_bytree': 0.3494501800394404, 'reg_alpha': 8.155447911151738, 'reg_lambda': 1.0139295380033555, 'min_child_weight': 8}

Tuning LightGBM (50 trials)


Best trial: 7. Best value: 0.632919: 100%|██████████| 50/50 [04:07<00:00,  4.95s/it]


LightGBM done in 0:04:07.549170
  Best MAPE: 0.6329%
  Best params: {'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.06333268775321843, 'num_leaves': 30, 'subsample': 0.9010984903770198, 'colsample_bytree': 0.35218545057583955, 'reg_alpha': 7.620481786158549, 'reg_lambda': 0.08916674715636537, 'min_child_samples': 14}

Saved: saved_models/optuna_tuning_results.joblib


In [7]:
summary_rows = []
for model_name, res in TUNING_RESULTS.items():
    row = {"Model": model_name, "Best_MAPE": round(res["best_value"], 4)}
    row.update(res["best_params"])
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("saved_models/optuna_tuning_summary.csv", index=False)
display(summary_df)
print("Saved: saved_models/optuna_tuning_summary.csv")


,Model,Best_MAPE,n_estimators,max_depth,max_features,max_samples,min_samples_split,min_samples_leaf,learning_rate,subsample,colsample_bytree,reg_alpha,reg_lambda,min_child_weight,num_leaves,min_child_samples
0,RandomForest,0.6336,800,3,log2,0.7,9.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ExtraTrees,0.6329,500,5,log2,NaN,4.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,XGBoost,0.6329,300,11,NaN,NaN,NaN,NaN,0.014782,0.855154,0.349450,8.155448,1.013930,8.0,NaN,NaN
3,LightGBM,0.6329,400,6,NaN,NaN,NaN,NaN,0.063333,0.901098,0.352185,7.620482,0.089167,NaN,30.0,14.0


Saved: saved_models/optuna_tuning_summary.csv
